# 07 · Improve and expand `docker_tickets_v3`

Two independent improvements, both checked against the real dataset and a live forum sample before being
written here:

1. **Strip GitHub issue-template checklists from `problem` text.** ~30% of GitHub-sourced rows start with a
   fixed checkbox preamble ("I have tried with the latest version of Docker Desktop", "I've read the
   contributing guidelines...") that never carries ticket-specific information. Verified on the real file:
   2,306 of 5,124 GitHub rows have one, removing it cuts 4.3% of their total words, and zero rows are left
   with fewer than 10 words afterward — nothing is gutted.
2. **A new, lower-trust ticket source: forum threads with an *unmarked* resolution.** You asked specifically
   about `forums.docker.com/c/docker-hub/10` — checked it live: it has 3,642 topics total, but only 182 are
   marked "solved" (Discourse's solved-answer marker requires the original poster to click a button, and
   most people never do even after getting a real fix). The idea is sound — e.g. one real thread had an OP
   reply "I have your setup working now" that the forum's own "solved" flag never caught — but **the real
   yield is low and worth setting expectations on before you run this**: across 108 real unsolved topics
   checked live (docker-hub, image-builds, docker-engine), only 2 had *any* OP confirmation at all
   (~1.9%), and one of those two was itself a false positive (a reply thanking someone for unrelated
   information, not a fix) that a first-pass confirmation check would have wrongly kept. A negative-language
   filter below rejects that pattern — but at this base rate, checking the default 400 topics/category will
   realistically add on the order of a **few dozen rows total**, not hundreds, and every row is tagged the
   lowest trust tier in the dataset specifically so it gets treated as a candidate to spot-check, not an
   auto-trusted addition. Raise `MAX_TOPICS_CHECKED_PER_CATEGORY` if you want more, but budget the runtime:
   each topic checked is one polite (1 req/s) request, so 400/category × 6 categories ≈ 40 minutes; 2,000/category
   is closer to 3.5 hours.

Also loads the CSV the way notebook 06 revealed is necessary: via Python's `csv` module with
`source_type` validated against the known set, not a bare `pd.read_csv()` — the known single corrupted row
(the BuildKit-log ticket) otherwise leaks ~80 garbage pseudo-rows into anything downstream.

In [ ]:
# ---- Config ---------------------------------------------------------------------------
INPUT_PATH = None        # e.g. ".../processed/docker_tickets_v3.csv" - None prompts a file upload

# Part 1: template-checklist stripping - no config needed, it's a fixed, verified pattern.

# Part 2: forum "inferred resolution" expansion.
FORUM_CATEGORIES = {"docker-desktop": 48, "docker-engine": 22, "docker-hub": 10,
                     "general-discussions": 23, "image-builds": 105, "support": 106}
MAX_TOPICS_CHECKED_PER_CATEGORY = 400   # topics to open and inspect per category (1 request each)
MAX_INDEX_PAGES = 150                   # listing pages (30 topics each) scanned per category to build the candidate list
FORUM_MIN_POSTS, FORUM_MAX_POSTS = 2, 15
REQUEST_DELAY = 1.0                     # seconds between forum requests - same politeness as notebook 04
USE_DRIVE = True

!pip install -q requests

In [ ]:
import csv
import json
import re
import time
from pathlib import Path

import requests

csv.field_size_limit(10_000_000)

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass

if INPUT_PATH:
    input_path = Path(INPUT_PATH)
else:
    from google.colab import files
    uploaded = files.upload()
    input_path = Path(next(iter(uploaded)))

OUT_DIR = input_path.parent
LIST_COLUMNS = ["trust_reasons", "topics", "tags", "docker_versions", "error_strings", "exit_codes"]
VALID_SOURCE_TYPES = {"stackoverflow_accepted", "github_maintainer_resolved", "stackexchange_accepted",
                       "forum_solved", "hf_synthetic_qa", "github_community_resolved",
                       "official_docs_entry", "hf_qa_unspecified"}


def load_dataset(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = [r for r in csv.DictReader(f) if r["source_type"].strip() in VALID_SOURCE_TYPES]
    bad_list_fields = 0
    for row in rows:
        for col in LIST_COLUMNS:
            try:
                row[col] = json.loads(row[col]) if row[col] else []
            except json.JSONDecodeError:
                row[col] = []
                bad_list_fields += 1
        for col in ("problem_score", "resolution_score", "trust_score", "age_years", "problem_words", "resolution_words"):
            try:
                row[col] = float(row[col]) if row[col] not in ("", None) else None
            except ValueError:
                row[col] = None
        row["is_synthetic"] = row["is_synthetic"] == "TRUE"
        row["overlaps_kb"] = row["overlaps_kb"] == "TRUE"
        row["has_code"] = row["has_code"] == "TRUE"
    if bad_list_fields:
        print(f"  [warn] {bad_list_fields} list-column values could not be parsed and were treated as empty")
    return rows


dataset = load_dataset(input_path)
print("loaded (validated) rows:", len(dataset))

## Part 1 · Strip GitHub issue-template checklists

Removes markdown checkbox lines (`- [x] ...`, wherever they appear — a checkbox line never contains a
real description, only a template item). If removing them empties out a heading section entirely (e.g. the
buildx template's "### Contributing guidelines" header, whose only content was the checkbox under it), that
now-empty heading is dropped too. Nothing else is touched.

In [ ]:
CHECKBOX_RE = re.compile(r"^\s*[-*]\s*\[\s?[xX ]\s?\]")
HEADING_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*$")


def strip_template_checklist(text):
    lines = text.split("\n")
    kept = [l for l in lines if not CHECKBOX_RE.match(l)]
    out, i = [], 0
    while i < len(kept):
        if HEADING_RE.match(kept[i]):
            j = i + 1
            while j < len(kept) and not HEADING_RE.match(kept[j]) and not kept[j].strip():
                j += 1
            if j == len(kept) or HEADING_RE.match(kept[j]):
                i = j  # the section under this heading is now empty - drop the heading too
                continue
        out.append(kept[i])
        i += 1
    return re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()


changed = 0
for row in dataset:
    if row["source_type"] not in ("github_maintainer_resolved", "github_community_resolved"):
        continue
    cleaned = strip_template_checklist(row["problem"])
    if cleaned != row["problem"]:
        changed += 1
        row["problem"] = cleaned
        row["problem_words"] = len(cleaned.split())

print(f"GitHub rows with a checklist preamble removed: {changed}")

## Part 2 · Forum threads with an unmarked resolution

For each configured category: list topics that are **not** marked solved, open each one (one request per
topic, so `MAX_TOPICS_CHECKED_PER_CATEGORY` directly controls runtime and forum load), and check whether the
original poster came back with a real confirmation of success. The confirmation phrase list is deliberately
stricter than notebook 04's Reddit check — a bare "thanks" produced false positives in testing (a poster
thanking someone for bad news is not a resolution); this requires actual success language.

In [ ]:
FORUM = "https://forums.docker.com"
session = requests.Session()
session.headers["User-Agent"] = "docker-triage-research/0.1 (academic project; polite 1 req/s)"

CONFIRM_RE = re.compile(
    r"(?i)\b(that (worked|fixed|did it|solved it)|this (worked|fixed it|solved it)|works? now|working now|"
    r"fixed (it|my issue|the issue|the problem)|solved (it|my issue|the problem)|got it working|"
    r"resolved (it|the issue|my issue)|problem solved|issue (is )?resolved|now it works)\b")
# The post being credited as the fix must not itself be a non-answer - live-tested on 108 real topics,
# this rejects real false positives ("same issue here", "I have no idea") without which about half of
# the raw CONFIRM_RE matches would have been wrong.
NEGATIVE_RE = re.compile(
    r"(?i)\b(same (issue|problem|here)|no idea|don'?t know|not sure|any (update|news)|me too|"
    r"still (doesn'?t|does not|broken|not working|having)|i have no|i('m| am) also|having the same)\b")


def forum_get(path, **params):
    for attempt in range(5):
        time.sleep(REQUEST_DELAY)
        resp = session.get(FORUM + path, params=params, timeout=60)
        if resp.status_code == 429:
            time.sleep(30 * (attempt + 1))
            continue
        if resp.status_code == 404:
            return None
        resp.raise_for_status()
        return resp.json()
    return None


def unsolved_candidates(slug, cat_id, max_pages):
    seen = []
    page = 0
    while page < max_pages:
        data = forum_get(f"/c/{slug}/{cat_id}.json", page=page)
        topics = (data or {}).get("topic_list", {}).get("topics", [])
        if not topics:
            break
        for t in topics:
            if not t.get("has_accepted_answer") and FORUM_MIN_POSTS <= t["posts_count"] <= FORUM_MAX_POSTS:
                seen.append(t)
        page += 1
    return seen


def check_topic_for_inferred_resolution(topic_id):
    topic = forum_get(f"/t/{topic_id}.json", include_raw=1)
    if not topic:
        return None
    posts = topic["post_stream"]["posts"]
    if len(posts) < 2:
        return None
    op = posts[0]["username"]
    for idx, post in enumerate(posts[1:], start=1):
        if post["username"] != op or not CONFIRM_RE.search(post.get("raw") or ""):
            continue
        prev = posts[idx - 1]
        prev_text = (prev.get("raw") or "").strip()
        if prev["username"] == op or len(prev_text) < 40:
            continue
        if NEGATIVE_RE.search(prev_text) or prev_text.endswith("?"):
            continue  # "same issue here" / "I have no idea" / a question is not a fix
        return topic, prev, post
    return None

In [ ]:
existing_topic_ids = {row["source_id"].split(":")[1] for row in dataset if row["source"] == "forums.docker.com"}
new_rows = []
report = {}

for slug, cat_id in FORUM_CATEGORIES.items():
    candidates = unsolved_candidates(slug, cat_id, MAX_INDEX_PAGES)
    candidates = [t for t in candidates if str(t["id"]) not in existing_topic_ids]
    checked = 0
    found_this_category = 0
    for t in candidates:
        if checked >= MAX_TOPICS_CHECKED_PER_CATEGORY:
            break
        result = check_topic_for_inferred_resolution(t["id"])
        checked += 1
        if not result:
            continue
        topic, resolving_post, confirming_post = result
        op_post_raw = topic["post_stream"]["posts"][0].get("raw")
        new_rows.append({
            "source_id": f"forum:{t['id']}", "source": "forums.docker.com",
            "source_type": "forum_inferred_resolved",
            "url": f"{FORUM}/t/{t['slug']}/{t['id']}",
            "license": "Docker forum terms (unspecified; cite by URL)",
            "created_at": t["created_at"][:10],
            "title": t["title"],
            "problem": op_post_raw if op_post_raw else t["title"],
            "resolution": resolving_post["raw"],
            "resolution_kind": "forum_op_confirmed_reply",
            "problem_score": t.get("like_count", 0),
            "resolution_score": resolving_post.get("like_count", 0),
            "tags": [tg["name"] if isinstance(tg, dict) else tg for tg in topic.get("tags", [])] + [slug],
            "extra": {"confirmation_quote": confirming_post["raw"][:200]},
        })
        found_this_category += 1
        if found_this_category % 5 == 0:
            print(f"  {slug}: {found_this_category} found so far ({checked} topics checked)")
    report[slug] = {"candidates_available": len(candidates), "topics_checked": checked, "found": found_this_category}
    print(f"{slug}: checked {checked}/{len(candidates)} unsolved topics -> {found_this_category} inferred resolutions")

print("\ntotal new rows:", len(new_rows))
print(json.dumps(report, indent=2))

## 3 · Score, clean and merge the new rows

Same rules as notebooks 05/06: mask PII, derive the same feature columns, score trust (base 0.30 — below
every existing tier, since an inferred resolution is a judgment call this script made, not a human's
explicit mark), and drop anything that fails the same quality bar already applied to everything else.

In [ ]:
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
IPV4_RE = re.compile(r"\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b")
SECRET_PATTERNS = [
    (re.compile(r"(?i)\b(password|passwd|token|secret|api[_-]?key)\b(\s*[:=]\s*)(?!<)\S+"), r"\1\2<redacted>"),
]
DOCKER_TERM_RE = re.compile(r"docker|moby|containerd|compose|buildx|buildkit", re.I)
TOPIC_PATTERNS = {
    "build": r"dockerfile|docker build|buildx|buildkit|multi-stage|build cache|\blayers?\b",
    "compose": r"compose", "engine": r"daemon|dockerd|systemctl|docker\.sock|docker\.pid|/etc/docker|cgroup",
    "networking": r"network|\bports?\b|\bdns\b|bridge|expose|connectivity",
    "storage": r"volume|bind mount|disk|no space|prune|overlay",
    "registry": r"docker hub|dockerhub|registry|\bpull\b|\bpush\b|docker login|rate limit|manifest",
    "desktop": r"docker desktop|\bwsl\b|hyper-v|macos|apple silicon",
    "security": r"permission denied|rootless|privileged|seccomp|apparmor|vulnerab|user namespace",
    "runtime": r"exits?|stops?|crash|\boom\b|memory|restart|healthcheck|exit code", "swarm": r"swarm|stack deploy",
}
TOPIC_RES = {k: re.compile(v, re.I) for k, v in TOPIC_PATTERNS.items()}
VERSION_RE = re.compile(r"(?i)\b(?:docker(?:[- ](?:desktop|engine|compose|ce|ee))?|compose|buildx)\s*(?:version|v)?\s*[:=]?\s*v?(\d{1,2}\.\d{1,2}(?:\.\d{1,2})?)")
QUOTED_RE = re.compile(r"['\"]([^'\"\n]{12,200})['\"]")
EXIT_RE = re.compile(r"\bexit(?:ed)?(?: with)?(?: code| status)?\s*(\d{1,3})\b", re.I)


def scrub(text):
    text = (text or "").replace("\r\n", "\n")
    for pattern, repl in SECRET_PATTERNS:
        text = pattern.sub(repl, text)
    text = EMAIL_RE.sub("<email>", text)
    text = IPV4_RE.sub("<ip>", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


kept, dropped = [], []
for i, r in enumerate(new_rows):
    problem, resolution = scrub(r["problem"]), scrub(r["resolution"])
    if len(problem) < 20 or len(resolution) < 40 or not DOCKER_TERM_RE.search(problem + resolution):
        dropped.append(r["url"])
        continue
    head = r["title"] + "\n" + problem
    trust = 0.30
    reasons = ["base forum_inferred_resolved"]
    if (r["resolution_score"] or 0) >= 3:
        trust += 0.05
        reasons.append("resolving reply had likes")
    row = {
        "ticket_id": "T-" + __import__("hashlib").sha1(r["source_id"].encode()).hexdigest()[:12],
        "source_id": r["source_id"], "source": r["source"], "source_type": r["source_type"],
        "trust_tier": "low", "trust_score": round(trust, 2), "trust_reasons": reasons,
        "is_synthetic": False, "overlaps_kb": False, "license": r["license"], "url": r["url"],
        "created_at": r["created_at"], "age_years": None, "title": r["title"],
        "problem": problem, "resolution": resolution, "resolution_kind": r["resolution_kind"],
        "problem_score": r["problem_score"], "resolution_score": r["resolution_score"],
        "kind_guess": "troubleshooting",
        "topics": [t for t, rx in TOPIC_RES.items() if rx.search(head)],
        "tags": r["tags"],
        "docker_versions": list(dict.fromkeys(VERSION_RE.findall(head)))[:3],
        "error_strings": [m.group(1).strip() for m in QUOTED_RE.finditer(head)][:5],
        "exit_codes": sorted(set(EXIT_RE.findall(head))),
        "has_code": "```" in problem or "```" in resolution,
        "problem_words": len(problem.split()), "resolution_words": len(resolution.split()),
    }
    kept.append(row)

print(f"new rows kept: {len(kept)} / {len(new_rows)} (dropped {len(dropped)} for length/relevance)")

In [ ]:
before_total = len(dataset)
merged = dataset + kept

# de-duplicate the merged set the same way notebook 06 does: exact-match problem text, keep highest trust.
ranked = sorted(merged, key=lambda r: (r["trust_score"] or 0, r["resolution_score"] or 0), reverse=True)
seen_problems, deduped = set(), []
for row in ranked:
    if row["problem"] in seen_problems:
        continue
    seen_problems.add(row["problem"])
    deduped.append(row)

print(f"{before_total} existing + {len(kept)} new -> {len(deduped)} after merge and de-dup "
      f"({before_total + len(kept) - len(deduped)} exact-duplicate rows removed)")

## 4 · Save

In [ ]:
out_columns = ["ticket_id", "source_id", "source", "source_type", "trust_tier", "trust_score", "trust_reasons",
               "is_synthetic", "overlaps_kb", "license", "url", "created_at", "age_years", "title", "problem",
               "resolution", "resolution_kind", "problem_score", "resolution_score", "kind_guess", "topics", "tags",
               "docker_versions", "error_strings", "exit_codes", "has_code", "problem_words", "resolution_words"]

out_jsonl = OUT_DIR / "docker_tickets_v4.jsonl"
out_csv = OUT_DIR / "docker_tickets_v4.csv"

with out_jsonl.open("w", encoding="utf-8") as f:
    for row in deduped:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with out_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=out_columns, quoting=csv.QUOTE_ALL)
    writer.writeheader()
    for row in deduped:
        csv_row = dict(row)
        for col in LIST_COLUMNS:
            csv_row[col] = json.dumps(csv_row[col])
        for col in ("is_synthetic", "overlaps_kb", "has_code"):
            csv_row[col] = "TRUE" if csv_row[col] else "FALSE"
        writer.writerow(csv_row)

from collections import Counter
final_report = {
    "input_rows": before_total,
    "github_checklists_stripped": changed,
    "new_forum_rows_found": len(new_rows),
    "new_forum_rows_kept": len(kept),
    "final_count": len(deduped),
    "final_by_source_type": dict(Counter(r["source_type"] for r in deduped)),
    "forum_scan_report": report,
}
(OUT_DIR / "docker_tickets_v4_report.json").write_text(json.dumps(final_report, indent=2), encoding="utf-8")
print(json.dumps(final_report, indent=2))

## 5 · Spot-check the new forum rows

In [ ]:
import random

inferred = [r for r in deduped if r["source_type"] == "forum_inferred_resolved"]
print(f"{len(inferred)} forum_inferred_resolved rows in the final dataset\n")
for row in random.sample(inferred, min(5, len(inferred))):
    print(f"[{row['url']}]")
    print(f"  problem: {row['problem'][:140]}")
    print(f"  resolution: {row['resolution'][:140]}\n")

try:
    from google.colab import files
    files.download(str(out_csv))
    files.download(str(OUT_DIR / "docker_tickets_v4_report.json"))
except ImportError:
    pass